<a href="https://colab.research.google.com/github/BandanaSingha24/Integrated_Multiomic_Survival_Model/blob/main/02_Feature_Engineering/processed_Clinical_Engineering_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
df=pd.read_csv("md-1_processed_clinical_data.csv")

print(df.head())

  PATIENT_ID  LYMPH_NODES_EXAMINED_POSITIVE    NPI CELLULARITY CHEMOTHERAPY  \
0    MB-0000                           10.0  6.044         NaN           NO   
1    MB-0002                            0.0  4.020        High           NO   
2    MB-0005                            1.0  4.030        High          YES   
3    MB-0006                            3.0  4.050    Moderate          YES   
4    MB-0008                            8.0  6.080        High          YES   

   COHORT   ER_IHC HER2_SNP6 HORMONE_THERAPY INFERRED_MENOPAUSAL_STATE  ...  \
0     1.0  Positve   NEUTRAL             YES                      Post  ...   
1     1.0  Positve   NEUTRAL             YES                       Pre  ...   
2     1.0  Positve   NEUTRAL             YES                       Pre  ...   
3     1.0  Positve   NEUTRAL             YES                       Pre  ...   
4     1.0  Positve   NEUTRAL             YES                      Post  ...   

                        CANCER_TYPE_DETAILED ER_ST

In [4]:
#step 1:Encoding and Scaling

from sklearn.preprocessing import StandardScaler
import pandas as pd

# 1. Binary encoding for clear two-state variables
binary_cols = ['CHEMOTHERAPY', 'HORMONE_THERAPY']
for col in binary_cols:
    df[col] = df[col].map({'YES': 1, 'NO': 0})

# 2. One-hot encoding for multi-class categorical variables
categorical_cols = ['CELLULARITY', 'HER2_STATUS', 'INFERRED_MENOPAUSAL_STATE']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# 3. Scale numerical features
numerical_cols = ['LYMPH_NODES_EXAMINED_POSITIVE', 'NPI']
scaler = StandardScaler()
df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])

# Preview your engineered clinical dataset
print(df_encoded.head())

  PATIENT_ID  LYMPH_NODES_EXAMINED_POSITIVE       NPI  CHEMOTHERAPY  COHORT  \
0    MB-0000                       2.003916  1.695121           0.0     1.0   
1    MB-0002                      -0.485579 -0.007391           0.0     1.0   
2    MB-0005                      -0.236630  0.001020           1.0     1.0   
3    MB-0006                       0.261269  0.017844           1.0     1.0   
4    MB-0008                       1.506017  1.725402           1.0     1.0   

    ER_IHC HER2_SNP6  HORMONE_THERAPY     SEX INTCLUST  ...  ONCOTREE_CODE  \
0  Positve   NEUTRAL              1.0  Female     4ER+  ...            IDC   
1  Positve   NEUTRAL              1.0  Female     4ER+  ...            IDC   
2  Positve   NEUTRAL              1.0  Female        3  ...            IDC   
3  Positve   NEUTRAL              1.0  Female        9  ...           MDLC   
4  Positve   NEUTRAL              1.0  Female        9  ...           MDLC   

   PR_STATUS SAMPLE_TYPE TUMOR_SIZE TUMOR_STAGE TMB_NONS

In [5]:
# step 2: One-hot encoding and Scaling againg for thoe categorial data which has not converted into numeric data

# 1. One-hot encode the remaining categorical text columns
final_encoded_cols = ['SEX', 'ONCOTREE_CODE', 'PR_STATUS', 'SAMPLE_TYPE']
df_final = pd.get_dummies(df_encoded, columns=final_encoded_cols, drop_first=True)

# 2. Scale the remaining raw numeric column
df_final['TUMOR_SIZE'] = scaler.fit_transform(df_final[['TUMOR_SIZE']])

# 3. Verify all columns are now strictly numeric (excluding PATIENT_ID)
print(df_final.dtypes)
print("\nShape of finalized clinical data:", df_final.shape)

PATIENT_ID                        object
LYMPH_NODES_EXAMINED_POSITIVE    float64
NPI                              float64
CHEMOTHERAPY                     float64
COHORT                           float64
ER_IHC                            object
HER2_SNP6                         object
HORMONE_THERAPY                  float64
INTCLUST                          object
AGE_AT_DIAGNOSIS                 float64
OS_MONTHS                        float64
OS_STATUS                         object
CLAUDIN_SUBTYPE                   object
THREEGENE                         object
VITAL_STATUS                      object
LATERALITY                        object
RADIO_THERAPY                     object
HISTOLOGICAL_SUBTYPE              object
BREAST_SURGERY                    object
RFS_MONTHS                       float64
RFS_STATUS                        object
SAMPLE_ID                         object
CANCER_TYPE                       object
CANCER_TYPE_DETAILED              object
ER_STATUS       

In [6]:
# step 2:Final data cleaning and Numeric encoding

# 1. Drop redundant text identifiers and uniform columns
cols_to_drop = ['SAMPLE_ID', 'CANCER_TYPE', 'CANCER_TYPE_DETAILED']
df_cleaned = df_final.drop(columns=[col for col in cols_to_drop if col in df_final.columns])

# 2. Convert remaining object columns to dummies
final_objects = df_cleaned.select_dtypes(include=['object']).columns.tolist()
if 'PATIENT_ID' in final_objects:
    final_objects.remove('PATIENT_ID') # Keep PATIENT_ID as string for merging

df_fully_numeric = pd.get_dummies(df_cleaned, columns=final_objects, drop_first=True)

# 3. Final Verification Check
print("Remaining non-numeric columns:")
print(df_fully_numeric.select_dtypes(exclude=['number', 'bool']).columns.tolist())

Remaining non-numeric columns:
['PATIENT_ID']


In [8]:
from google.colab import files

df_fully_numeric.to_csv('md-2_processed_clinical_engineering_data.csv', index=False)

files.download('md-2_processed_clinical_engineering_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>